# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:

%pip install -q duckdb huggingface_hub pandas

import os
import getpass
import duckdb
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    print(name, con.execute(f"SELECT COUNT(*) FROM {src}").fetchdf().iloc[0, 0])

df = con.execute(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 5").fetchdf()
df.head()


Note: you may need to restart the kernel to use updated packages.
dim_clients 104
dim_content 519606
fact_daily 78835655
fact_query_90d 2414248


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row= one content page (content_id) 

In [11]:
import pandas as pd

df = pd.read_csv(r"d:\git\Assignment-1\data\raw\content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Unique client_id: {df['client_id'].nunique():,}")
print("Duplicate content_id rows:", df.duplicated(subset=['content_id']).sum())

print("\nWindow & filters:")
print("min content_age_days:", df['content_age_days'].min())
print("min impressions_90d:", df['impressions_90d'].min())
print("rows with impressions_90d > 0:", (df['impressions_90d'] > 0).sum())
print("rows with content_age_days >= 90:", (df['content_age_days'] >= 90).sum())
print("fraction meeting both filters:", ((df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)).mean())

print("\nLabel / proxy:")
print(df['trend_direction'].value_counts(dropna=False))
if 'is_declining_label' in df.columns:
    print("is_declining_label mean (rate):", df['is_declining_label'].mean())
    mismatch = (df['is_declining_label'].fillna(0).astype(int) != (df['trend_direction'].str.lower() == 'down').astype(int)).sum()
    print("is_declining_label vs trend_direction mismatches:", mismatch)

print("\nExample rows:")
print(df[['content_id','client_id','content_age_days','impressions_90d','trend_direction']].head().to_string(index=False))

print("\nAny date-like columns present:", [c for c in df.columns if 'date' in c])

Rows: 30,000
Unique content_id: 30,000
Unique client_id: 32
Duplicate content_id rows: 0

Window & filters:
min content_age_days: 90
min impressions_90d: 1
rows with impressions_90d > 0: 30000
rows with content_age_days >= 90: 30000
fraction meeting both filters: 1.0

Label / proxy:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Example rows:
          content_id         client_id  content_age_days  impressions_90d trend_direction
content_304f48230142 client_f369cb89fc               187             3803            down
content_a1fb4e703a9e client_4e07408562               445            15320            down
content_9aa793d4d895 client_7f2253d7e2               141            12581            down
content_331d6c4de07b client_19581e27de               463            11751          stable
content_d99b7a2d90ca client_3fdba35f04               263            19140            down

Any date-like columns present: ['days_

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: (observable signals before a refresh decision)
impressions_90d, clicks_90d, sessions_90d, impressions_last_30d, impressions_prev_30d, ctr, avg_position, search_volume, word_count,days_since_last_update, ai_sessions_90d.

Label:
trend_direction, trend_pct, is_declining_label (used only as target not features to prevent leakage).

Context: content_id, client_id, content_type, main_intent (for grouping only).

Excluded: trend_direction is excluded as feature because it is the label source


In [2]:
import pandas as pd
df = pd.read_csv(r"d:\git\Assignment-1\data\raw\content_refresh_anonymized.csv")  

print("columns:", df.columns.tolist())
print("\nlabel distribution:\n", df['trend_direction'].value_counts(dropna=False))
cols = ['impressions_90d','clicks_90d','sessions_90d','impressions_last_30d','impressions_prev_30d',
        'ctr','avg_position','search_volume','word_count','days_since_last_update','ai_sessions_90d']
print("\npresence & missingness (sample features):")
for c in cols:
    print(f"{c:25} present={c in df.columns:5}  null_frac={df[c].isna().mean() if c in df.columns else 'NA'}")
print("\nunique content_id:", df['content_id'].nunique(), "rows:", len(df))
print("duplicate content_id rows:", df.duplicated(subset=['content_id']).sum())
print("\nmin content_age_days:", df['content_age_days'].min())
print("min impressions_90d:", df['impressions_90d'].min())
print("rows with impressions_90d>0:", (df['impressions_90d']>0).sum())
print("rows with content_age_days>=90:", (df['content_age_days']>=90).sum())

columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

label distribution:
 trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

presence & missingness (sample features):
impressions_90d           present

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every contract claim above must be backed by a query. Each bullet below is a claim plus the check i ran

Grain: confirm one row = one content_id without any duplicates

Window & Filter: confirm dataset filters/ snapshot assumptions; content_age_days >= 90, impression_90d > 0.

Label: show trend_direction distribution, trend_pct missingness , and confirm is_declining_label matches trend_direction == "down"

Missingness: show null fraction for key feature columns and check systematic missingness by content_type.

Avg position note: report avg_position == 0 count; 0 means "no position data", treat as missing. 

Volume/usability filters: counts of rows at impression thresholds (100, 25, 500, 3000) to decide minimum volume rules.

Client coverage: rows per client_id to inspect unbalanced history.

In [3]:
# 1) Grain
print("Rows:", len(df))
print("Unique content_id:", df['content_id'].nunique())
print("Duplicate content_id rows (should be 0):", df.duplicated(subset=['content_id']).sum())

# 2) Window & filters
print("\nWindow & filters:")
print("min content_age_days:", df['content_age_days'].min())
print("min impressions_90d:", df['impressions_90d'].min())
print("rows with impressions_90d > 0:", (df['impressions_90d'] > 0).sum())
print("rows with content_age_days >= 90:", (df['content_age_days'] >= 90).sum())
print("fraction meeting both filters:", ((df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)).mean())

# 3) Label
print("\nLabel / proxy:")
print(df['trend_direction'].value_counts(dropna=False))
print("trend_pct missing fraction:", df['trend_pct'].isna().mean() if 'trend_pct' in df.columns else 'no trend_pct')
if 'is_declining_label' in df.columns:
    print("is_declining_label mean (rate):", df['is_declining_label'].mean())
    mismatch = (df['is_declining_label'].fillna(0).astype(int) != (df['trend_direction'].str.lower() == 'down').astype(int)).sum()
    print("is_declining_label vs trend_direction mismatches:", mismatch)

# 4) Missingness (key features)
features = ['impressions_90d','clicks_90d','sessions_90d','impressions_last_30d','impressions_prev_30d',
            'ctr','avg_position','search_volume','word_count','days_since_last_update','ai_sessions_90d']
present = [c for c in features if c in df.columns]
print("\nMissingness (fraction null) for selected features:")
print(df[present].isna().mean().sort_values(ascending=False))

# 5) Missingness by content_type
if 'content_type' in df.columns:
    print("\nMissingness by content_type (sample):")
    sample_cols = present[:6]
    print(df.groupby('content_type')[sample_cols].apply(lambda g: g.isna().mean()).round(3).head(10))

# 6) avg_position note
if 'avg_position' in df.columns:
    print("\navg_position == 0 count (means no position data):", (df['avg_position'] == 0).sum())

# 7) Volume thresholds
thresholds = [0, 100, 250, 500, 3000]
print("\nRows by impressions_90d thresholds:")
for t in thresholds:
    print(f" imps >= {t:4}: {(df['impressions_90d'] >= t).sum():,}")

# 8) Client coverage
print("\nRows per client (top 10):")
print(df['client_id'].value_counts().head(10))

# 9) Example rows
print("\nExample rows (first 5):")
print(df[['content_id','client_id','content_age_days','impressions_90d','trend_direction']].head().to_string(index=False))

Rows: 30000
Unique content_id: 30000
Duplicate content_id rows (should be 0): 0

Window & filters:
min content_age_days: 90
min impressions_90d: 1
rows with impressions_90d > 0: 30000
rows with content_age_days >= 90: 30000
fraction meeting both filters: 1.0

Label / proxy:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
trend_pct missing fraction: 0.11293333333333333

Missingness (fraction null) for selected features:
word_count                0.256633
search_volume             0.082267
impressions_90d           0.000000
sessions_90d              0.000000
clicks_90d                0.000000
impressions_last_30d      0.000000
impressions_prev_30d      0.000000
avg_position              0.000000
ctr                       0.000000
days_since_last_update    0.000000
ai_sessions_90d           0.000000
dtype: float64

Missingness by content_type (sample):
                    impressions_90d  clicks_90d  sessions_90d  \

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can support page level refresh opportunity analysis, but it cannot tell the full product decision process or exact true future outcomes. The rows are a snapshot of trailing 90-day signals only, not full daily histories, so we cannot reconstruct exact daily behavior within that window. The data is also filtered to pages with impressions_90d > 0 and content_age_days >= 90, meaning zero-impression or very new pages are absent. Some dimensions are unbalanced across clients, and GA4/GSC coverage is not uniform, so the dataset cannot tell us equally about every client or every page history. Missingness is systematic, especially in keyword/context fields, and avg_position == 0 means "no position data" not "position zero". The label is a proxy based on recent trend direction, so it is useful for refresh scoring, but it is not a true future outcome label and should be treated as a decision support signal rather than a final business truth.

In [5]:
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Rows with impressions_90d > 0:", (df["impressions_90d"] > 0).sum())
print("Rows with content_age_days >= 90:", (df["content_age_days"] >= 90).sum())
# 1) Filtered dataset: zero-impression/new pages are absent
print("\nFiltered rows check:")
print("Rows excluded by impressions filter:", (df["impressions_90d"] <= 0).sum())
print("Rows excluded by age filter:", (df["content_age_days"] < 90).sum())
# 2) Unbalanced history / client coverage
print("\nClient coverage:")
print(df["client_id"].value_counts().head(10))
# 3) Missingness / systematic gaps
feature_cols = ["search_volume", "competition", "cpc", "word_count", "avg_position", "ai_sessions_90d"]
print("\nMissingness for selected fields:")
print(df[feature_cols].isna().mean().sort_values(ascending=False))
# 4) avg_position special case
print("\navg_position == 0 count (means no position data):", (df["avg_position"] == 0).sum())
# 5) Trend proxy label limitation
print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts(dropna=False))
print("trend_pct missing fraction:", df["trend_pct"].isna().mean() if "trend_pct" in df.columns else "no trend_pct")
# 6) Example low-volume pages and sparse fields
print("\nSparse AI sessions count:", (df["ai_sessions_90d"] > 0).sum())
print("Zero-session rows:", (df["sessions_90d"] == 0).sum())
# 7) Quick sanity check for window filters
print("\nSnapshot constraints check:")
print("min content_age_days:", df["content_age_days"].min())
print("min impressions_90d:", df["impressions_90d"].min())

Rows: 30000
Unique content_id: 30000
Rows with impressions_90d > 0: 30000
Rows with content_age_days >= 90: 30000

Filtered rows check:
Rows excluded by impressions filter: 0
Rows excluded by age filter: 0

Client coverage:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031
Name: count, dtype: int64

Missingness for selected fields:
word_count         0.256633
search_volume      0.082267
competition        0.082267
cpc                0.082267
avg_position       0.000000
ai_sessions_90d    0.000000
dtype: float64

avg_position == 0 count (means no position data): 1205

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
trend_pct missing fraction: 0.11293333333333333

Spars

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.